# Step 5: Orchestrate comparison and explain the result

Select the **.venv** kernel and run from the top. We reuse the saved **synthetic**
inputs from Notebook 04; its kernel does not need to stay running. No API calls
are made. If its report is missing, run Notebook 04 through Section 9 first.

This graph connects the pricing engine to LangGraph. Its explanation is generated
by Python from calculated values, not by another model. We are building the
comparison portion of the optimizer; real PDF terms are not yet pricing inputs.

## 1. Import the workflow
As in Notebook 03, nodes exchange shared state. This graph has three successful
steps: validate inputs, calculate costs, explain results. Invalid inputs take a
separate failure path.

In [1]:
from pathlib import Path
import json
from decimal import Decimal
from copy import deepcopy
from html import escape
from IPython.display import display, HTML
from electricity_optimizer.comparison_workflow import build_comparison_workflow

PROJECT_ROOT = Path.cwd()
source_path = PROJECT_ROOT / "output" / "lesson04_synthetic" / "SYNTHETIC_comparison.json"
if not source_path.is_file():
    raise FileNotFoundError("Run Notebook 04 through Section 9 to save its synthetic report first.")

def show_rows(headers, rows):
    def row(values, tag):
        return "<tr>" + "".join(f"<{tag} style='padding:6px 12px;text-align:left'>{escape(str(v))}</{tag}>" for v in values) + "</tr>"
    display(HTML("<table>" + row(headers, "th") + "".join(row(r, "td") for r in rows) + "</table>"))

## 2. Reuse inputs, recompute outputs
Read usage and fictional plan definitions from the saved report. We deliberately
recalculate the bills instead of trusting its saved totals. The synthetic flag
must be true, and at least two uniquely named demo plans are required.

In [2]:
saved = json.loads(source_path.read_text(encoding="utf-8"))
comparison_inputs = {
    "synthetic": saved.get("synthetic"),
    "usage": saved["usage"],
    "plans": [entry["plan"] for entry in saved["comparisons"]],
}
print("Usage months:", len(comparison_inputs["usage"]["months"]))
print("Fictional plans:", [p["name"] for p in comparison_inputs["plans"]])

Usage months: 12
Fictional plans: ['Fictional Simple', 'Fictional Threshold']


## 3. Build the graph
`compile()` creates a runnable graph. The conditional route is rule-based; it sends
invalid inputs to `failed` before calculations. These nodes are ordinary Python
functions, not multiple AI agents.

```mermaid
flowchart LR
    A[Validate inputs] -->|Valid| B[Calculate costs]
    A -->|Invalid| F[Failed]
    B --> C[Explain results]
```

You can inspect the exact graph wiring in `electricity_optimizer/comparison_workflow.py`.

In [3]:
comparison_graph = build_comparison_workflow()
print(comparison_graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	validate_inputs(validate_inputs)
	calculate_costs(calculate_costs)
	explain_results(explain_results)
	failed(failed)
	__end__([<p>__end__</p>]):::last
	__start__ --> validate_inputs;
	calculate_costs --> explain_results;
	validate_inputs -.-> calculate_costs;
	validate_inputs -.-> failed;
	explain_results --> __end__;
	failed --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 4. Watch the graph run
Each streamed state includes the latest trace and status. `comparison_complete`
means this fictional calculation finished; it does not approve any real contract.

In [4]:
comparison_state = None
for state in comparison_graph.stream({"inputs": comparison_inputs}, stream_mode="values"):
    if state.get("trace"):
        print(state["trace"][-1], "->", state["status"])
    comparison_state = state

validate_inputs -> validated
calculate_costs -> calculated
explain_results -> comparison_complete


## 5. Read the explanation and check its evidence
The explanation names the cheapest plan (or all tied cheapest plans), the annual
cost difference, and the months receiving credits. Each claim is generated from
monthly results. The table below preserves the bill components used for the totals.

In [5]:
if comparison_state["status"] == "failed":
    print(comparison_state["error"])
else:
    print(comparison_state["explanation"])
    show_rows(["Fictional plan", "Annual USD"],
        [(r.plan.name, r.annual_usd) for r in comparison_state["results"]])
    show_rows(["Plan", "Month", "kWh", "Energy $", "Base $", "Delivery $", "Credit $", "Total $"],
        [(r.plan.name, b.month, b.kwh, b.energy_usd, b.base_usd, b.delivery_usd, b.credit_usd, b.total_usd)
         for r in comparison_state["results"] for b in r.bills])

Synthetic teaching comparison only; not a recommendation for a real household.

Fictional Simple has the lowest annual cost at $2,214.90, $99.10 less than the next cheapest plan, Fictional Threshold.

Fictional Simple: $2,214.90 annually; $0.00 in credits across 0 months (none).

Fictional Threshold: $2,314.00 annually; $200.00 in credits across 4 months (2025-06, 2025-07, 2025-08, 2025-09).

Totals include energy, base and delivery charges, less credits. Taxes and switching fees are excluded. Rates are fixed for all 12 months; components round to cents half up and credits cannot exceed the monthly subtotal. Different usage can change the ranking.


Fictional plan,Annual USD
Fictional Simple,2214.90
Fictional Threshold,2314.00


Plan,Month,kWh,Energy $,Base $,Delivery $,Credit $,Total $
Fictional Simple,2025-01,850,102.00,10.00,47.50,0.00,159.50
Fictional Simple,2025-02,720,86.40,10.00,41.00,0.00,137.40
Fictional Simple,2025-03,650,78.00,10.00,37.50,0.00,125.50
Fictional Simple,2025-04,700,84.00,10.00,40.00,0.00,134.00
Fictional Simple,2025-05,900,108.00,10.00,50.00,0.00,168.00
Fictional Simple,2025-06,1200,144.00,10.00,65.00,0.00,219.00
Fictional Simple,2025-07,1500,180.00,10.00,80.00,0.00,270.00
Fictional Simple,2025-08,1600,192.00,10.00,85.00,0.00,287.00
Fictional Simple,2025-09,1250,150.00,10.00,67.50,0.00,227.50
Fictional Simple,2025-10,950,114.00,10.00,52.50,0.00,176.50


## 6. Experiment: rerun with 20% higher usage
Make a separate input copy, adjust consumption, and run the same graph. No nodes
need changing. This is a uniform hypothetical scenario, not a forecast. Compare the
credited months with the base case to understand why the ranking can change.

In [6]:
scenario_inputs = deepcopy(comparison_inputs)
for month in scenario_inputs["usage"]["months"]:
    month["kwh"] = str(Decimal(month["kwh"]) * Decimal("1.20"))
scenario_state = comparison_graph.invoke({"inputs": scenario_inputs})
print(scenario_state["explanation"] or scenario_state["error"])

Synthetic teaching comparison only; not a recommendation for a real household.

Fictional Threshold has the lowest annual cost at $2,592.80, $29.08 less than the next cheapest plan, Fictional Simple.

Fictional Threshold: $2,592.80 annually; $400.00 in credits across 8 months (2025-01, 2025-05, 2025-06, 2025-07, 2025-08, 2025-09, 2025-10, 2025-12).

Fictional Simple: $2,621.88 annually; $0.00 in credits across 0 months (none).

Totals include energy, base and delivery charges, less credits. Taxes and switching fees are excluded. Rates are fixed for all 12 months; components round to cents half up and credits cannot exceed the monthly subtotal. Different usage can change the ranking.


## 7. Watch invalid input stop the workflow
Remove one month from a copy. The graph should go directly from validation to
failure without calculating or explaining a winner. The base result is unchanged.

In [7]:
invalid_inputs = deepcopy(comparison_inputs)
invalid_inputs["usage"]["months"].pop()
invalid_state = comparison_graph.invoke({"inputs": invalid_inputs})
print("Trace:", " -> ".join(invalid_state["trace"]))
print("Status:", invalid_state["status"])
print("Problem:", invalid_state["error"])
assert invalid_state["results"] == []
assert invalid_state["explanation"] is None

Trace: validate_inputs -> failed
Status: failed
Problem: Invalid comparison inputs (ValidationError): 1 validation error for UsageYear
months
  Tuple should have at least 12 items after validation, not 11 [type=too_short, input_value=[{'month': '2025-01', 'kw...2025-11', 'kwh': '750'}], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/too_short


## 8. Save the base workflow report
Save the original scenario only, with input definitions, monthly evidence, trace
and explanation. The 20% scenario and failed experiment do not replace the base
result. Decimal amounts remain strings in JSON. Rerunning replaces this report.

In [8]:
if comparison_state["status"] != "comparison_complete":
    print("No successful base comparison to save.")
else:
    report = {
        "synthetic": True,
        "source_report": str(source_path),
        "inputs": comparison_inputs,
        "status": comparison_state["status"],
        "trace": comparison_state["trace"],
        "explanation": comparison_state["explanation"],
        "comparisons": [r.model_dump(mode="json") for r in comparison_state["results"]],
    }
    target_dir = PROJECT_ROOT / "output" / "lesson05_synthetic"
    target_dir.mkdir(parents=True, exist_ok=True)
    target = target_dir / "SYNTHETIC_workflow_comparison.json"
    target.write_text(json.dumps(report, indent=2), encoding="utf-8")
    assert json.loads(target.read_text(encoding="utf-8")) == report
    print("Saved:", target)

Saved: c:\Users\Nyalo\VSCode_Projects\Electricity_Agreement_Optimizer\output\lesson05_synthetic\SYNTHETIC_workflow_comparison.json


## What did we connect?
Notebook 03 handles PDF evidence and review routing. Notebook 05 adds a reusable
LangGraph comparison workflow around the calculation engine. They remain separate
until a reviewed-pricing conversion step can safely connect actual agreement terms
to numeric rules. Missing fees must not become zero by default.

**Pause and explain:** which node rejects 11 months? Where do the credited months
in the explanation come from? Why does changing input usage require no graph edits?

Next we can build that explicit reviewed-input boundary, then connect comparison
and explanation to the supervisor. Time-of-use rates, taxes, tiers and switching
costs remain outside the current fictional fixed-rate engine.

[LangGraph graph API](https://docs.langchain.com/oss/python/langgraph/graph-api) explains
the state, node, and edge concepts used here.